In [4]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

In [8]:
# 1) LLM (OpenAI)
# Pick a model you have access to. "gpt-4o-mini" is commonly used for structured JSON tasks.
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=api_key)


In [9]:
# 2) Schema (Pydantic)
class Book(BaseModel):
    """Information about a book."""
    title: str = Field(description="The title of the book")
    author: str = Field(description="The author of the book")
    year_of_publication: int = Field(description="The year the book was published")

# 3) Parser + format instructions
parser = JsonOutputParser(pydantic_object=Book)
format_instructions = parser.get_format_instructions()


In [12]:
# 4) Prompt
template = ChatPromptTemplate.from_messages([
    ("system", "You output ONLY valid JSON that matches the schema exactly. No backticks."),
    ("human",
     "Generate JSON about the user input according to the format instructions.\n"
     "Input: {input}\n"
     "{format_instructions}"
    )
])

# 5) Chain
chain = template.partial(format_instructions=format_instructions) | llm | parser

# Single call
print(chain.invoke({"input": "East of Eden"}))

# Batch call
book_titles = ["Dune", "Neuromancer", "Snow Crash", "The Left Hand of Darkness", "Foundation"]
print(chain.batch([{"input": t} for t in book_titles]))


{'title': 'East of Eden', 'author': 'John Steinbeck', 'year_of_publication': 1952}
[{'title': 'Dune', 'author': 'Frank Herbert', 'year_of_publication': 1965}, {'title': 'Neuromancer', 'author': 'William Gibson', 'year_of_publication': 1984}, {'title': 'Snow Crash', 'author': 'Neal Stephenson', 'year_of_publication': 1992}, {'title': 'The Left Hand of Darkness', 'author': 'Ursula K. Le Guin', 'year_of_publication': 1969}, {'title': 'Foundation', 'author': 'Isaac Asimov', 'year_of_publication': 1951}]
